# Candidate Evaluation

This notebook evaluates **sampled candidate responses** from a `candidate_bank.jsonl` file against the BioASQ gold list answers.

It uses the repo's **official BioASQ exact list scoring logic** from `src.utility.bioasq_format.py`, not the `cse_dpo` semantic matcher.

What it computes for each sampled candidate:

- list precision, recall, and F1 from the official BioASQ Java scorer
- number of correct entities
- number of incorrect additions
- number of omitted gold entities
- answer-set cardinality
- exact-normalized parsed item counts
- entity-set distance to gold
- evidence support for each predicted entity

Important note:

- the Java evaluator is the official aggregate scorer, but it reports dataset-level results rather than convenient per-candidate rows
- this notebook therefore uses the repo's per-question **official-aligned exact scorer** for candidate-level analysis and avoids the semantic DPO matcher entirely
- `policy_probability_difference` is still **not stored** in the candidate bank, so this notebook marks it unavailable


In [ ]:
from __future__ import annotations

import argparse
import copy
import json
import sys
from pathlib import Path
from dataclasses import replace
from types import SimpleNamespace

import pandas as pd
from IPython.display import Markdown, display


def find_task_root(start: Path) -> Path:
    current = start.resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "Artifacts").exists():
            return candidate
        nested = candidate / "Task-Structured Counterfactual Preference Mining"
        if (nested / "src").exists() and (nested / "Artifacts").exists():
            return nested
    raise FileNotFoundError("Could not locate the Task-Structured Counterfactual Preference Mining root.")


TASK_ROOT = find_task_root(Path.cwd())
if str(TASK_ROOT) not in sys.path:
    sys.path.insert(0, str(TASK_ROOT))

from cse_dpo.construct_set_edit_pairs import candidate_is_evidence_supported
from cse_dpo.match_gold_groups import load_candidate_bank_records, load_question_examples
from cse_dpo.normalize_set_answers import normalize_answer_surface, normalize_evidence_text, parse_list_output
from src.utility.bioasq_format import exact_answer_groups, match_to_gold_group, parse_prediction_items
from src.utility.data import clean_text
from src.utility.eval_types import EvalExample

pd.options.display.max_colwidth = 200
pd.options.display.max_rows = 200

print(f"TASK_ROOT = {TASK_ROOT}")


In [ ]:
TRAIN_FILES = [TASK_ROOT / "data" / "training13b.json"]
AVAILABLE_CANDIDATE_BANKS = sorted((TASK_ROOT / "Artifacts").glob("**/candidate_bank.jsonl"))
AVAILABLE_CANDIDATE_BANKS = sorted((TASK_ROOT / "Artifacts" / "Study4" / "candidate_banks" / "full_gold_model_full_gold_data").glob("**/candidate_bank.jsonl"))
DEFAULT_CANDIDATE_INPUT = AVAILABLE_CANDIDATE_BANKS[0] if AVAILABLE_CANDIDATE_BANKS else None

QUESTION_INPUT = TRAIN_FILES
CANDIDATE_INPUT = DEFAULT_CANDIDATE_INPUT
DATASET_NAME = "bioasq"
ALLOW_FALLBACK_SPLIT = False
ALLOW_FALLBACK_PAIR_CONSTRUCTION = False
ONLY_PAIR_ELIGIBLE = False
QUESTION_IDS: set[str] | None = None
MAX_RECORDS = None
USE_CACHE = True
PROGRESS_EVERY = 500
SHOW_TOP_N = 20

print("Available candidate banks:")
for path in AVAILABLE_CANDIDATE_BANKS:
    print(f"  - {path.relative_to(TASK_ROOT)}")

print("\nActive configuration:")
print(f"  QUESTION_INPUT = {[str(path.relative_to(TASK_ROOT)) for path in QUESTION_INPUT]}")
print(f"  CANDIDATE_INPUT = {None if CANDIDATE_INPUT is None else CANDIDATE_INPUT.relative_to(TASK_ROOT)}")
print(f"  ONLY_PAIR_ELIGIBLE = {ONLY_PAIR_ELIGIBLE}")
print(f"  ALLOW_FALLBACK_SPLIT = {ALLOW_FALLBACK_SPLIT}")
print(f"  ALLOW_FALLBACK_PAIR_CONSTRUCTION = {ALLOW_FALLBACK_PAIR_CONSTRUCTION}")
print(f"  QUESTION_IDS = {QUESTION_IDS}")
print(f"  MAX_RECORDS = {MAX_RECORDS}")
print(f"  USE_CACHE = {USE_CACHE}")
print(f"  PROGRESS_EVERY = {PROGRESS_EVERY}")
print("  Note: set MAX_RECORDS = None to score the full candidate bank.")

if CANDIDATE_INPUT is None:
    raise FileNotFoundError("No candidate_bank.jsonl file was found under Artifacts.")


In [ ]:
from hashlib import md5
from time import perf_counter


def normalize_nonempty_items(items: tuple[str, ...]) -> list[str]:
    return [normalize_answer_surface(item) for item in items if normalize_answer_surface(item)]


def unique_preserving_order(values: list[str]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for value in values:
        if value in seen:
            continue
        seen.add(value)
        ordered.append(value)
    return ordered


def build_cache_path() -> Path:
    cache_dir = TASK_ROOT / "Artifacts" / "analysis_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    question_input_label = ",".join(str(path.resolve()) for path in QUESTION_INPUT)
    question_ids_label = ",".join(sorted(QUESTION_IDS)) if QUESTION_IDS else "all"
    payload = "|".join(
        [
            "candidates-evaluation-official-java-v2",
            str(CANDIDATE_INPUT.resolve()),
            question_input_label,
            question_ids_label,
            str(ONLY_PAIR_ELIGIBLE),
            str(ALLOW_FALLBACK_SPLIT),
            str(ALLOW_FALLBACK_PAIR_CONSTRUCTION),
            str(MAX_RECORDS),
        ]
    )
    cache_key = md5(payload.encode("utf-8")).hexdigest()[:16]
    return cache_dir / f"candidates_eval_official_java_{cache_key}.pkl"


def load_raw_list_examples(paths: list[Path]) -> dict[str, EvalExample]:
    examples: dict[str, EvalExample] = {}
    for path in paths:
        payload = json.loads(path.read_text(encoding="utf-8"))
        questions = payload.get("questions") if isinstance(payload, dict) else None
        if not isinstance(questions, list):
            raise ValueError(f"Expected raw BioASQ JSON with a 'questions' list: {path}")
        for question in questions:
            if not isinstance(question, dict):
                continue
            if clean_text(question.get("type", "")).lower() != "list":
                continue
            question_id = clean_text(question.get("id", ""))
            if not question_id or question_id in examples:
                continue
            examples[question_id] = EvalExample(
                question_id=question_id,
                question_type="list",
                body=clean_text(question.get("body", "")),
                instruction="",
                resources=(),
                gold_output="",
                source_path=str(path),
                raw_question=question,
            )
    return examples


def official_list_match_details(example: EvalExample, prediction_text: str) -> dict[str, object]:
    gold_groups = exact_answer_groups(example, "list")
    predicted_items = parse_prediction_items(prediction_text, "list")
    matched_gold = set()
    matched_gold_group_ids: list[int] = []
    entity_details = []

    for item in predicted_items:
        matched_gold_group_id = None
        for gold_index, gold_group in enumerate(gold_groups):
            if gold_index in matched_gold:
                continue
            if match_to_gold_group(item, gold_group):
                matched_gold.add(gold_index)
                matched_gold_group_id = gold_index
                matched_gold_group_ids.append(gold_index)
                break

        entity_details.append(
            {
                "surface": item,
                "matched_gold_group_id": matched_gold_group_id,
                "label": "gold_matched" if matched_gold_group_id is not None else "metric_negative",
                "match_type": "exact_surface_diagnostic" if matched_gold_group_id is not None else None,
                "match_confidence": 1.0 if matched_gold_group_id is not None else 0.0,
            }
        )

    true_positives = len(matched_gold_group_ids)
    prediction_count = len(predicted_items)
    gold_count = len(gold_groups)
    false_positives = prediction_count - true_positives
    false_negatives = gold_count - true_positives
    precision = true_positives / prediction_count if prediction_count else 0.0
    recall = true_positives / gold_count if gold_count else 0.0
    f1 = 0.0 if precision + recall == 0 else (2 * precision * recall) / (precision + recall)

    return {
        "predicted_items": predicted_items,
        "gold_groups": gold_groups,
        "matched_gold_group_ids": matched_gold_group_ids,
        "missing_gold_group_ids": [index for index in range(gold_count) if index not in matched_gold],
        "entity_details": entity_details,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "prediction_count": prediction_count,
        "gold_count": gold_count,
        "correct_entities": true_positives,
        "incorrect_additions": false_positives,
        "omitted_gold_entities": false_negatives,
    }


QUESTION_OBJECTS_BY_ID = load_question_examples(paths=QUESTION_INPUT, dataset_name=DATASET_NAME)
OFFICIAL_EXAMPLES_BY_ID = load_raw_list_examples(list(QUESTION_INPUT))
RECORDS = load_candidate_bank_records(
    paths=[CANDIDATE_INPUT],
    questions_by_id=QUESTION_OBJECTS_BY_ID,
    dataset_name=DATASET_NAME,
)

RECORDS = [record for record in RECORDS if record.question_id in OFFICIAL_EXAMPLES_BY_ID]
if QUESTION_IDS is not None:
    RECORDS = [record for record in RECORDS if record.question_id in QUESTION_IDS]
if MAX_RECORDS is not None:
    RECORDS = RECORDS[:MAX_RECORDS]

CACHE_PATH = build_cache_path()

print(f"Cache path = {CACHE_PATH.relative_to(TASK_ROOT)}")
print(f"Questions with official gold loaded = {len(OFFICIAL_EXAMPLES_BY_ID)}")
print(f"Candidate-bank records prepared = {len(RECORDS)}")

def score_candidate_records_with_official_java(records):
    prediction_rows = []
    examples_by_key = {}
    record_ids = {}
    for record_index, record in enumerate(records, start=1):
        source_example = OFFICIAL_EXAMPLES_BY_ID[record.question_id]
        synthetic_id = f'candidate_{record_index:08d}_{record.question_id}'
        raw_question = copy.deepcopy(source_example.raw_question)
        raw_question['id'] = synthetic_id
        example = replace(source_example, question_id=synthetic_id, raw_question=raw_question)
        examples_by_key[(synthetic_id, 'list')] = example
        record_ids[record_index] = synthetic_id
        prediction_rows.append({
            'question_id': synthetic_id, 'question_type': 'list',
            'body': example.body, 'source_path': source_example.source_path,
            'prediction': record.raw_output,
        })
    payload = evaluate_with_bioasq_java(
        prediction_rows=prediction_rows, examples_by_key=examples_by_key,
        model_label='candidate-bank-analysis',
        model_dir=CACHE_PATH.parent / (CACHE_PATH.stem + '_official'),
        args=argparse.Namespace(
            bioasq_java_jar=str(TASK_ROOT / 'third_party/Evaluation-Measures/flat/BioASQEvaluation/dist/BioASQEvaluation.jar'),
            bioasq_java_heap='2G', bioasq_java_version=5),
        include_per_question=True,
    )
    by_id = {row['question_id']: row for row in payload['per_question']}
    return {index: by_id[synthetic_id] for index, synthetic_id in record_ids.items()}


if USE_CACHE and CACHE_PATH.exists():
    cache_payload = pd.read_pickle(CACHE_PATH)
    RESPONSE_DF = cache_payload["response_df"].copy()
    ENTITY_DF = cache_payload["entity_df"].copy()
    print(f"Loaded cached tables from {CACHE_PATH.relative_to(TASK_ROOT)}")
    print(f"RESPONSE_DF shape = {RESPONSE_DF.shape}; ENTITY_DF shape = {ENTITY_DF.shape}")
else:
    OFFICIAL_SCORES_BY_RECORD_INDEX = score_candidate_records_with_official_java(RECORDS)
    start_time = perf_counter()
    response_rows = []
    entity_rows = []
    kept_records = 0
    total_records = len(RECORDS)

    for record_index, record in enumerate(RECORDS, start=1):
        example = OFFICIAL_EXAMPLES_BY_ID[record.question_id]
        parsed = parse_list_output(record.raw_output, allow_fallback_split=ALLOW_FALLBACK_SPLIT)
        pair_eligible = parsed.status == "ok" or (ALLOW_FALLBACK_PAIR_CONSTRUCTION and parsed.status == "fallback_split")
        pair_exclusion_reason = None if pair_eligible else (parsed.status or "unknown_parser_status")
        if ONLY_PAIR_ELIGIBLE and not pair_eligible:
            if record_index % PROGRESS_EVERY == 0 or record_index == total_records:
                print(f"Processed {record_index}/{total_records} records; kept {kept_records} pair-eligible rows.")
            continue

        kept_records += 1
        diagnostics = official_list_match_details(example, record.raw_output)
        official_score = OFFICIAL_SCORES_BY_RECORD_INDEX[record_index]
        official = diagnostics
        raw_nonempty_items = normalize_nonempty_items(tuple(official["predicted_items"]))
        raw_exact_unique_items = unique_preserving_order(raw_nonempty_items)
        evidence_question = SimpleNamespace(evidence=tuple(record.evidence))
        evidence_text = normalize_evidence_text(record.evidence)
        evidence_lookup = {
            "evidence_text": evidence_text,
            "evidence_text_compact": evidence_text.replace(" ", ""),
        }

        evidence_support_count = 0
        unsupported_surfaces: list[str] = []
        evidence_support_details = []

        for item_index, entity in enumerate(official["entity_details"]):
            evidence_supported = candidate_is_evidence_supported(evidence_question, entity["surface"], evidence_lookup)
            evidence_support_count += int(evidence_supported)
            if not evidence_supported:
                unsupported_surfaces.append(entity["surface"])
            entity_rows.append(
                {
                    "question_id": record.question_id,
                    "sample_id": record.sample_id,
                    "response_id": record.response_id,
                    "question_text": example.body,
                    "surface": entity["surface"],
                    "normalized": normalize_answer_surface(entity["surface"]),
                    "label": entity["label"],
                    "matched_gold_group_id": entity["matched_gold_group_id"],
                    "match_type": entity["match_type"],
                    "match_confidence": entity["match_confidence"],
                    "duplicate_count": 1,
                    "first_index": item_index,
                    "evidence_supported": evidence_supported,
                }
            )
            evidence_support_details.append(
                {
                    "surface": entity["surface"],
                    "normalized": normalize_answer_surface(entity["surface"]),
                    "label": entity["label"],
                    "matched_gold_group_id": entity["matched_gold_group_id"],
                    "match_type": entity["match_type"],
                    "evidence_supported": evidence_supported,
                }
            )

        response_rows.append(
            {
                "question_id": record.question_id,
                "sample_id": record.sample_id,
                "response_id": record.response_id,
                "question_text": example.body,
                "parser_status": parsed.status,
                "pair_eligible": pair_eligible,
                "pair_exclusion_reason": pair_exclusion_reason,
                "raw_output": record.raw_output,
                "parsed_items": list(parsed.items),
                "prediction_items": list(official["predicted_items"]),
                "matched_gold_group_ids": list(official["matched_gold_group_ids"]),
                "missing_gold_group_ids": list(official["missing_gold_group_ids"]),
                "precision": official_score["precision"],
                "recall": official_score["recall"],
                "f1": official_score["f1"],
                "correct_entities": official["correct_entities"],
                "incorrect_additions": official["incorrect_additions"],
                "omitted_gold_entities": official["omitted_gold_entities"],
                "answer_set_cardinality": official["prediction_count"],
                "gold_count": official["gold_count"],
                "invalid_addition_rate": 1.0 - official_score["precision"] if official["prediction_count"] else 0.0,
                "valid_omission_rate": 1.0 - official_score["recall"] if official["gold_count"] else 0.0,
                "entity_set_distance_to_gold": official["incorrect_additions"] + official["omitted_gold_entities"],
                "raw_parsed_item_count": len(parsed.items),
                "raw_nonempty_item_count": len(raw_nonempty_items),
                "raw_exact_normalized_unique_count": len(raw_exact_unique_items),
                "raw_exact_normalized_unique_items": raw_exact_unique_items,
                "has_duplicate_exact_normalized_predictions": len(raw_exact_unique_items) != len(raw_nonempty_items),
                "generated_token_count": record.generated_token_count,
                "evidence_supported_prediction_count": evidence_support_count,
                "evidence_unsupported_prediction_count": official["prediction_count"] - evidence_support_count,
                "all_predictions_evidence_supported": evidence_support_count == official["prediction_count"],
                "unsupported_prediction_surfaces": unsupported_surfaces,
                "evidence_support_details": evidence_support_details,
                "scoring_backend": "bioasq_java",
                "policy_probability_difference_available": False,
                "policy_probability_difference": None,
            }
        )

        if record_index % PROGRESS_EVERY == 0 or record_index == total_records:
            print(f"Processed {record_index}/{total_records} records; kept {kept_records} response rows.")

    RESPONSE_DF = pd.DataFrame(response_rows)
    ENTITY_DF = pd.DataFrame(entity_rows)
    elapsed_seconds = perf_counter() - start_time
    print(f"Built {len(RESPONSE_DF)} response rows and {len(ENTITY_DF)} entity rows in {elapsed_seconds:.1f}s.")
    if USE_CACHE:
        pd.to_pickle({"response_df": RESPONSE_DF, "entity_df": ENTITY_DF}, CACHE_PATH)
        print(f"Saved cache to {CACHE_PATH.relative_to(TASK_ROOT)}")


In [ ]:
summary_row = {
    "candidate_bank": CANDIDATE_INPUT.relative_to(TASK_ROOT).as_posix(),
    "responses": len(RESPONSE_DF),
    "pair_eligible_responses": int(RESPONSE_DF["pair_eligible"].sum()) if not RESPONSE_DF.empty else 0,
    "malformed_responses": int((RESPONSE_DF["parser_status"] == "malformed").sum()) if not RESPONSE_DF.empty else 0,
    "avg_precision": RESPONSE_DF["precision"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_recall": RESPONSE_DF["recall"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_f1": RESPONSE_DF["f1"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_correct_entities": RESPONSE_DF["correct_entities"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_incorrect_additions": RESPONSE_DF["incorrect_additions"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_omitted_gold_entities": RESPONSE_DF["omitted_gold_entities"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_answer_set_cardinality": RESPONSE_DF["answer_set_cardinality"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_entity_set_distance_to_gold": RESPONSE_DF["entity_set_distance_to_gold"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_generated_token_count": RESPONSE_DF["generated_token_count"].dropna().mean() if not RESPONSE_DF.empty else 0.0,
    "avg_evidence_supported_prediction_count": RESPONSE_DF["evidence_supported_prediction_count"].mean() if not RESPONSE_DF.empty else 0.0,
    "avg_evidence_unsupported_prediction_count": RESPONSE_DF["evidence_unsupported_prediction_count"].mean() if not RESPONSE_DF.empty else 0.0,
    "responses_with_all_predictions_supported": int(RESPONSE_DF["all_predictions_evidence_supported"].sum()) if not RESPONSE_DF.empty else 0,
    "scoring_backend": "bioasq_java",
    "policy_probability_difference_available": False,
}

SUMMARY_DF = pd.DataFrame([summary_row])
display(SUMMARY_DF)

metric_columns = [
    "precision",
    "recall",
    "f1",
    "correct_entities",
    "incorrect_additions",
    "omitted_gold_entities",
    "answer_set_cardinality",
    "entity_set_distance_to_gold",
    "generated_token_count",
    "evidence_supported_prediction_count",
    "evidence_unsupported_prediction_count",
]

display(RESPONSE_DF[metric_columns].describe().transpose())


## Candidate-Bank Diversity

This section measures diversity at the **question level** for the active candidate bank.

It distinguishes three increasingly strict notions of diversity:

- `unique_raw_outputs`: exact raw-text variation from generation
- `unique_ordered_answer_lists`: normalized predicted items, keeping item order
- `unique_unordered_answer_sets`: normalized predicted items, ignoring item order but preserving duplicate multiplicity

The last metric is the most relevant one for whole-response DPO pair construction, because the pair pipeline deduplicates normalized answers very close to this level.


In [ ]:
import matplotlib.pyplot as plt


def normalized_prediction_key(prediction_items: object, *, ignore_order: bool) -> tuple[str, ...]:
    if not isinstance(prediction_items, list):
        return tuple()
    normalized_items = [
        normalized
        for item in prediction_items
        if (normalized := normalize_answer_surface(item))
    ]
    if ignore_order:
        normalized_items = sorted(normalized_items)
    return tuple(normalized_items)


def integer_bins(values: pd.Series) -> range:
    max_value = int(values.max()) if not values.empty else 0
    return range(1, max_value + 2)


DIVERSITY_DF = (
    RESPONSE_DF.assign(
        raw_output_key=RESPONSE_DF["raw_output"].fillna("").map(lambda value: clean_text(value) if isinstance(value, str) else ""),
        ordered_answer_key=RESPONSE_DF["prediction_items"].map(lambda items: normalized_prediction_key(items, ignore_order=False)),
        unordered_answer_key=RESPONSE_DF["prediction_items"].map(lambda items: normalized_prediction_key(items, ignore_order=True)),
    )
    .groupby(["question_id", "question_text"], as_index=False)
    .agg(
        samples=("response_id", "count"),
        pair_eligible_samples=("pair_eligible", "sum"),
        unique_raw_outputs=("raw_output_key", "nunique"),
        unique_ordered_answer_lists=("ordered_answer_key", "nunique"),
        unique_unordered_answer_sets=("unordered_answer_key", "nunique"),
        mean_f1=("f1", "mean"),
        max_f1=("f1", "max"),
        mean_answer_set_cardinality=("answer_set_cardinality", "mean"),
        mean_generated_token_count=("generated_token_count", "mean"),
    )
)

DIVERSITY_DF["raw_diversity_ratio"] = DIVERSITY_DF["unique_raw_outputs"] / DIVERSITY_DF["samples"].clip(lower=1)
DIVERSITY_DF["unordered_diversity_ratio"] = DIVERSITY_DF["unique_unordered_answer_sets"] / DIVERSITY_DF["samples"].clip(lower=1)

diversity_summary_row = {
    "candidate_bank": CANDIDATE_INPUT.relative_to(TASK_ROOT).as_posix(),
    "questions": len(DIVERSITY_DF),
    "avg_samples_per_question": DIVERSITY_DF["samples"].mean() if not DIVERSITY_DF.empty else 0.0,
    "avg_unique_raw_outputs": DIVERSITY_DF["unique_raw_outputs"].mean() if not DIVERSITY_DF.empty else 0.0,
    "avg_unique_ordered_answer_lists": DIVERSITY_DF["unique_ordered_answer_lists"].mean() if not DIVERSITY_DF.empty else 0.0,
    "avg_unique_unordered_answer_sets": DIVERSITY_DF["unique_unordered_answer_sets"].mean() if not DIVERSITY_DF.empty else 0.0,
    "median_unique_unordered_answer_sets": DIVERSITY_DF["unique_unordered_answer_sets"].median() if not DIVERSITY_DF.empty else 0.0,
    "questions_with_single_unordered_answer_set": int((DIVERSITY_DF["unique_unordered_answer_sets"] == 1).sum()) if not DIVERSITY_DF.empty else 0,
    "questions_with_4plus_unordered_answer_sets": int((DIVERSITY_DF["unique_unordered_answer_sets"] >= 4).sum()) if not DIVERSITY_DF.empty else 0,
    "avg_raw_diversity_ratio": DIVERSITY_DF["raw_diversity_ratio"].mean() if not DIVERSITY_DF.empty else 0.0,
    "avg_unordered_diversity_ratio": DIVERSITY_DF["unordered_diversity_ratio"].mean() if not DIVERSITY_DF.empty else 0.0,
}

display(pd.DataFrame([diversity_summary_row]))

display(
    DIVERSITY_DF[
        [
            "samples",
            "pair_eligible_samples",
            "unique_raw_outputs",
            "unique_ordered_answer_lists",
            "unique_unordered_answer_sets",
            "raw_diversity_ratio",
            "unordered_diversity_ratio",
            "mean_f1",
            "max_f1",
            "mean_answer_set_cardinality",
            "mean_generated_token_count",
        ]
    ].describe().transpose()
)

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(DIVERSITY_DF["unique_raw_outputs"], bins=integer_bins(DIVERSITY_DF["unique_raw_outputs"]), color="#4C78A8", edgecolor="white")
axes[0, 0].set_title("Unique raw outputs per question")
axes[0, 0].set_xlabel("Count")
axes[0, 0].set_ylabel("Questions")

axes[0, 1].hist(DIVERSITY_DF["unique_unordered_answer_sets"], bins=integer_bins(DIVERSITY_DF["unique_unordered_answer_sets"]), color="#F58518", edgecolor="white")
axes[0, 1].set_title("Unique unordered answer sets per question")
axes[0, 1].set_xlabel("Count")
axes[0, 1].set_ylabel("Questions")

axes[1, 0].scatter(DIVERSITY_DF["unique_raw_outputs"], DIVERSITY_DF["unique_unordered_answer_sets"], alpha=0.6, s=20, color="#54A24B")
max_axis_value = int(max(DIVERSITY_DF["unique_raw_outputs"].max(), DIVERSITY_DF["unique_unordered_answer_sets"].max())) if not DIVERSITY_DF.empty else 0
axes[1, 0].plot([0, max_axis_value], [0, max_axis_value], linestyle="--", linewidth=1, color="black")
axes[1, 0].set_title("Surface diversity vs set diversity")
axes[1, 0].set_xlabel("Unique raw outputs")
axes[1, 0].set_ylabel("Unique unordered answer sets")

axes[1, 1].scatter(DIVERSITY_DF["unique_unordered_answer_sets"], DIVERSITY_DF["mean_f1"], alpha=0.6, s=20, color="#E45756")
axes[1, 1].set_title("Set diversity vs mean official F1")
axes[1, 1].set_xlabel("Unique unordered answer sets")
axes[1, 1].set_ylabel("Mean F1")

plt.tight_layout()
plt.show()

LOW_DIVERSITY_QUESTIONS = DIVERSITY_DF.sort_values(
    ["unique_unordered_answer_sets", "unique_raw_outputs", "mean_f1"],
    ascending=[True, True, False],
).head(SHOW_TOP_N)

HIGH_DIVERSITY_QUESTIONS = DIVERSITY_DF.sort_values(
    ["unique_unordered_answer_sets", "unique_raw_outputs", "mean_f1"],
    ascending=[False, False, False],
).head(SHOW_TOP_N)

display(Markdown(f"### Lowest set-diversity questions (top {SHOW_TOP_N})"))
display(
    LOW_DIVERSITY_QUESTIONS[
        [
            "question_id",
            "question_text",
            "samples",
            "unique_raw_outputs",
            "unique_unordered_answer_sets",
            "mean_f1",
            "max_f1",
            "mean_answer_set_cardinality",
        ]
    ]
)

display(Markdown(f"### Highest set-diversity questions (top {SHOW_TOP_N})"))
display(
    HIGH_DIVERSITY_QUESTIONS[
        [
            "question_id",
            "question_text",
            "samples",
            "unique_raw_outputs",
            "unique_unordered_answer_sets",
            "mean_f1",
            "max_f1",
            "mean_answer_set_cardinality",
        ]
    ]
)
